# Tutorial 51: Manual Creation of Element Dataframes

This Example demonstrates the capabilities of the class Dataframes_SIR3S_Model that extends SIR3S_Model be abilities to work directley with pandas dataframes. It is shown how to create dataframes containing information about elements such as Nodes, Pipes, etc. existing in a SIR 3S Model. The methods presented are manual, user-defined and detailed. For creating more general dataframes with less input necessary, see Tutorial 52.   

# Toolkit Release

In [1]:
#pip install 

# Imports

## SIR 3S Toolkit

### Regular Import/Init

In [2]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [3]:
from sir3stoolkit.core import wrapper

In [4]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [5]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-11-25 16:22:29,412] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:22:29,412] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:22:29,432] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:22:29,601] INFO in sir3stoolkit.core.wrapper: [Initialization] Sir3S_Toolkit.dll file version 90.15.11.0 (matches this sir3stoolkit version).


### Additional Import/Init for Dataframes class

In [6]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [7]:
s3s = SIR3S_Model_Dataframes()

[2026-11-25 16:22:30,932] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


# Open Model

The tutorial uses *Network 3*, a fictional district heating network shared by the mantle tutorials: a heat plant (in its own container "Heizwerk"), about 180 consumers, a ring road and two longitudinal sections, calculated over one day. It is built from scratch with the SIR 3S Toolkit by [Network3.py](https://github.com/3SConsult/sir3stoolkit/blob/main/docs/source/networks/Network3.py); `Network3.db3` is included in the tutorial's ZIP archive.

Its elements have unique ID references (IDREFERENZ), e.g. `VL-HS-01` for the first supply pipe of the main street. They are used below to look up tks with [GetTkFromIDReference()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetTkFromIDReference) - the tks themselves differ between builds of the model.

In [8]:
s3s.OpenModel(dbName=r"Network3.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

[2026-11-25 16:22:35,894] INFO in sir3stoolkit.core.wrapper: Model is open for further operation


# Calculate Model

In [9]:
s3s.ExecCalculation(True) # To ensure result data

[2026-11-25 16:22:41,224] INFO in sir3stoolkit.core.wrapper: Model Calculation is complete


# Manual Dataframe creation

## Object type reference

Before building dataframes, it helps to know what an element type offers. [get_object_type_reference()](https://3sconsult.github.io/sir3stoolkit/modules.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.get_object_type_reference) looks an element type up in the object type reference shipped with sir3stoolkit - the same content as the [Object Types, Properties, and Result Value Types](https://3sconsult.github.io/sir3stoolkit/modules.html#object-types-properties-and-result-value-types) section of the API documentation. It needs no open model, and logs the reference's notice with every call.

In [10]:
node_reference = s3s.get_object_type_reference(s3s.ObjectTypes.Node)

[2026-11-25 16:22:41,238] INFO in sir3stoolkit.mantle.dataframes: [object type reference] Aggregated global view across configured network types. The below sections lists all table names from self.ObjectTypes_TableNames, along with their properties, result properties, and respective object type from self.ObjectTypes (needed for toolkit operations like self.InsertElement(), self.GetPropertiesOfElementType()). Result properties additionally list a Description and Standard Physical Unit where known, sourced from SIR 3S's own .MX1 output plus a handful of manual additions (see result_props_descriptions.ipynb). Both are in German, as embedded in SIR 3S itself. Use sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.get_result_file_properties to check what physical units were actually used in your local calculation, which can deviate from the standard physical units here. Each object type lists its Object Category - determined via the Toolkit API where possible, by hand otherwise, Unknown 

Besides the properties, it tells the table code, the object category, the toolkit method(s) to create an element of this type and the network types it can be inserted into:

In [11]:
{key: value for key, value in node_reference.items() if key not in ("properties", "result_properties")}

{'table_name': 'KNOT',
 'category': 'node',
 'create_with': ['SIR3S_Model.AddNewNode'],
 'insertable_in': ['Water', 'DistrictHeating', 'Gas', 'Steam'],
 'number_of_end_nodes': 0}

The properties with their value types (the first 10 of them):

In [12]:
dict(list(node_reference["properties"].items())[:10])

{'Beschreibung': 'string',
 'Fk2lknot': 'string',
 'Fkcont': 'string',
 'Fkfqps': 'string',
 'Fkfstf': 'string',
 'Fkpzon': 'string',
 'Fkutmp': 'string',
 'GeometriesDiffer': 'boolean',
 'HasBlockConnection': 'boolean',
 'Idreferenz': 'string'}

And the result properties with their description (in German) and unit, e.g. pressure, external flow and temperature:

In [13]:
{p: node_reference["result_properties"][p] for p in ["PH", "QM", "T"]}

{'PH': {'description': 'Druck', 'unit': '[bar]'},
 'QM': {'description': 'Externer Durchfluss', 'unit': '[m3/h]'},
 'T': {'description': 'Temperatur', 'unit': '[°C]'}}

**Note:** these units are SIR 3S's *standard* physical units. The units actually used in your local calculation can deviate from them - as the logged notice says, check them with [get_result_file_properties()](https://3sconsult.github.io/sir3stoolkit/modules.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.get_result_file_properties) (see Tutorial 57), which reads them from the calculation's result file.

The reference is a snapshot of SIR 3S 90-15. For the installed SIR 3S version, GetPropertiesofElementType() - used below - and GetResultProperties_from_elementType() remain authoritative.

## model_data - Nodes

We can use the [generate_element_model_data_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_element_model_data_dataframe) method to obtain a dataframe for all instances of a specifc component type (Nodes, Pipes, etc.) with user defined model_data properties (all non-result data related to a component) to be added. For this example we will use nodes.

In this case we are interested in Nodes, let's check what model_data properties are available for this element.

In [14]:
s3s.GetPropertiesofElementType(s3s.ObjectTypes.Node)

['Name',
 'Ktyp',
 'Zkor',
 'QmEin',
 'Lfakt',
 'Fkpzon',
 'Fkfstf',
 'Fkutmp',
 'Fkfqps',
 'Fkcont',
 'Fk2lknot',
 'Beschreibung',
 'Idreferenz',
 'Iplanung',
 'Kvr',
 'Qakt',
 'Xkor',
 'Ykor',
 'NodeNamePosition',
 'ShowNodeName',
 'KvrKlartext',
 'NumberOfVERB',
 'HasBlockConnection',
 'Tk',
 'Pk',
 'InVariant',
 'GeometriesDiffer',
 'SymbolFactor',
 'bz.Drakonz',
 'bz.Fk',
 'bz.Fkpvar',
 'bz.Fkqvar',
 'bz.Fklfkt',
 'bz.PhEin',
 'bz.Tm',
 'bz.Te',
 'bz.PhMin']

In the below subcases differnt choices of parameters for creating the model_data dataframe manually are shown. Note that not all possible combination of paramters are shown here.

### Subcase 1: Minimal

Let's say for now we only want a dataframe with all node tks without any related data.

In [15]:
(s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Node
                                        ,properties=[]
                                        )).head(3)

[2026-11-25 16:22:41,280] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node


[2026-11-25 16:22:41,283] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 592 element(s) of element type ObjectTypes.Node.


[2026-11-25 16:22:41,289] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 0 model_data properties.


[2026-11-25 16:22:41,289] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving ...


[2026-11-25 16:22:41,295] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (592, 1)


,tk
0,5431799190353317880
1,4876208276983667420
2,5141021320509852354


### Subcase 2: Specific Properties 

Let's say we now want the minimal dataframe extended with supply/return info (kvr) and height (Zkor) of the nodes.

In [16]:
(s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Node
                                        ,properties=["Kvr", "Zkor"]
                                        )).head(3)

[2026-11-25 16:22:41,308] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node


[2026-11-25 16:22:41,310] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 592 element(s) of element type ObjectTypes.Node.


[2026-11-25 16:22:41,312] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 2 model_data properties.


[2026-11-25 16:22:41,312] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Kvr', 'Zkor']...


[2026-11-25 16:22:41,351] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (592, 3)


,tk,Kvr,Zkor
0,5431799190353317880,1,198.09
1,4876208276983667420,2,198.09
2,5141021320509852354,1,197.95


### Subcase 3: Maximal

Let's say now we want basically all available model_data (non-result data) related to nodes. 

If geometry=True, the 2D Point geometry will be added for Nodes and it will be attempted to transform the DataFrame into a GeoDataFrame using the SRID (eg. EPSG: 25832) defined in the model. If no SRID is defined, the DataFrame will remain as a DataFrame, but the geometry column with be added nonetheless.

In [17]:
(s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Node
                                        ,properties=None # Instead of an empty or filled list, we pass None to indicate we want all available properties
                                        ,geometry=True # geometry data has to be explicitly requested
                                        )).head(3)

[2026-11-25 16:22:41,365] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node


[2026-11-25 16:22:41,367] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 592 element(s) of element type ObjectTypes.Node.


[2026-11-25 16:22:41,367] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.Node.


[2026-11-25 16:22:41,368] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 37 model_data properties.


[2026-11-25 16:22:41,368] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Name', 'Ktyp', 'Zkor', 'QmEin', 'Lfakt', 'Fkpzon', 'Fkfstf', 'Fkutmp', 'Fkfqps', 'Fkcont', 'Fk2lknot', 'Beschreibung', 'Idreferenz', 'Iplanung', 'Kvr', 'Qakt', 'Xkor', 'Ykor', 'NodeNamePosition', 'ShowNodeName', 'KvrKlartext', 'NumberOfVERB', 'HasBlockConnection', 'Tk', 'Pk', 'InVariant', 'GeometriesDiffer', 'SymbolFactor', 'bz.Drakonz', 'bz.Fk', 'bz.Fkpvar', 'bz.Fkqvar', 'bz.Fklfkt', 'bz.PhEin', 'bz.Tm', 'bz.Te', 'bz.PhMin'], geometry...


[2026-11-25 16:22:42,308] WARNING in sir3stoolkit.mantle.dataframes: [model_data] Spatial Reference Identifier (SRID) not defined in model. DataFrame cannot be transformed to GeoDataFrame but geometry column can be created independently of SRID. Returning regular DataFrame with a geometry column.


[2026-11-25 16:22:42,308] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (592, 39)


,tk,Name,Ktyp,Zkor,QmEin,Lfakt,Fkpzon,Fkfstf,Fkutmp,Fkfqps,...,bz.Drakonz,bz.Fk,bz.Fkpvar,bz.Fkqvar,bz.Fklfkt,bz.PhEin,bz.Tm,bz.Te,bz.PhMin,geometry
0,5431799190353317880,V-HS-00,QKON,198.09,0,1,-1,5233359644433413774,5363574168628413302,-1,...,0,5431799190353317880,-1,-1,-1,0,0,0,0,POINT Z (100 1000 198.08999633789062)
1,4876208276983667420,R-HS-00,QKON,198.09,0,1,-1,5233359644433413774,5363574168628413302,-1,...,0,4876208276983667420,-1,-1,-1,0,0,0,0,POINT Z (104 996 198.08999633789062)
2,5141021320509852354,V-HS-01,QKON,197.95,0,1,-1,5233359644433413774,5363574168628413302,-1,...,0,5141021320509852354,-1,-1,-1,0,0,0,0,POINT Z (200 1000 197.9499969482422)


### Subcase 4: Filtering + Element Type Col

Let's say we only want nodes inside a specific container and we also want to have an element type col adding "Node" to each row (useful when joining with other dataframes later).

In [18]:
for container_id in (s3s.GetTksofElementType(s3s.ObjectTypes.ObjectContainerSymbol)):
    print(f"{container_id}: {s3s.GetValue(container_id, 'Name')[0]}")

5586885184287337261: Network 3 - Musterstadt (fictional distr
5628722198385572227: Heizwerk


In [19]:
all_tks = s3s.GetTksofElementType(s3s.ObjectTypes.Node)

In [20]:
good_tks = []

In [21]:
main_container = s3s.GetMainContainer()[0]
for tk in all_tks:
    if s3s.GetValue(tk, "Fkcont")[0] == main_container: # not in the container "Heizwerk" (heat plant)
        if s3s.GetValue(tk, "Kvr")[0] == "1": # Supply/Vorlauf
            good_tks.append(tk)

In [22]:
(s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Node
                                        ,properties=[]
                                        ,tks=good_tks
                                        ,geometry=True
                                        ,element_type_col=True
                                        )).head(3)

[2026-11-25 16:22:42,401] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node


[2026-11-25 16:22:42,404] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 592 element(s) of element type ObjectTypes.Node.


[2026-11-25 16:22:42,405] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] 294 tks remain after filtering for given tks.


[2026-11-25 16:22:42,405] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 0 model_data properties.


[2026-11-25 16:22:42,405] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving geometry...


[2026-11-25 16:22:42,423] WARNING in sir3stoolkit.mantle.dataframes: [model_data] Spatial Reference Identifier (SRID) not defined in model. DataFrame cannot be transformed to GeoDataFrame but geometry column can be created independently of SRID. Returning regular DataFrame with a geometry column.


[2026-11-25 16:22:42,423] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (294, 3)


,tk,geometry,element type
0,5431799190353317880,POINT Z (100 1000 198.08999633789062),Node
1,5141021320509852354,POINT Z (200 1000 197.9499969482422),Node
2,4691128234547286647,POINT Z (300 1000 197.4199981689453),Node


## Result Data - Pipes

We can use the [generate_element_results_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_element_results_dataframe) method to obtain a dataframe for all instances of a specifc component type (Nodes, Pipes, etc.) with user defined result values for certain timestamps to be added. For this example we will use pipes.

If the methods for result retrieval presented here are too slow for you check out tutorial 57.

Let's check what result properties are available for Pipes.

In [23]:
s3s.GetResultProperties_from_elementType(elementType=s3s.ObjectTypes.Pipe
                                         ,onlySelectedVectors=False 
                                         )

['A',
 'ACALC',
 'CPI',
 'CPK',
 'DH',
 'DP',
 'DRAGRED',
 'DRAKONZ',
 'DSI',
 'DSK',
 'DTTR',
 'DWVERL',
 'DWVERLABS',
 'ETAAV',
 'FS',
 'HR',
 'HVEC',
 'IAKTIV',
 'IRTRENN',
 'JV',
 'JV2',
 'LAMBDA',
 'LECKEINAUS',
 'LECKMENGE',
 'LECKORT',
 'LINEPACK',
 'LINEPACKGEOM',
 'LINEPACKRATE',
 'MAINELEMENT',
 'MAV',
 'MI',
 'MK',
 'MKOND',
 'MMAX_INST',
 'MMIN_INST',
 'MVEC',
 'MVECMAX_INST',
 'MVECMIN_INST',
 'PAV',
 'PDAMPF',
 'PHR',
 'PHVEC',
 'PMAX',
 'PMIN',
 'PR',
 'PVEC',
 'PVECMAX_INST',
 'PVECMIN_INST',
 'QI2',
 'QK2',
 'QMAV',
 'QMI',
 'QMK',
 'QMMAX_INST',
 'QMMIN_INST',
 'QMVEC',
 'QSVB',
 'RHOAV',
 'RHOI',
 'RHOK',
 'RHOVEC',
 'SVEC',
 'TAV',
 'TI',
 'TK',
 'TTRVEC',
 'TVEC',
 'TVECMAX_INST',
 'TVECMIN_INST',
 'VAV',
 'VI',
 'VK',
 'VMAX_INST',
 'VMIN_INST',
 'VOLDA',
 'WALTERI',
 'WALTERK',
 'WVL',
 'ZAUS',
 'ZEIN',
 'ZHKNR',
 'ZVEC']

Let's also check what simulation timestamps are available in the model.

In [24]:
s3s.GetTimeStamps()[0] # list of simulation timestamps

['2026-01-15 00:00:00.000 +01:00',
 '2026-01-15 02:00:00.000 +01:00',
 '2026-01-15 04:00:00.000 +01:00',
 '2026-01-15 06:00:00.000 +01:00',
 '2026-01-15 08:00:00.000 +01:00',
 '2026-01-15 10:00:00.000 +01:00',
 '2026-01-15 12:00:00.000 +01:00',
 '2026-01-15 14:00:00.000 +01:00',
 '2026-01-15 16:00:00.000 +01:00',
 '2026-01-15 18:00:00.000 +01:00',
 '2026-01-15 20:00:00.000 +01:00',
 '2026-01-15 22:00:00.000 +01:00',
 '2026-01-16 00:00:00.000 +01:00']

In [25]:
s3s.GetTimeStamps()[1] # Static

'2026-01-15 00:00:00.000 +01:00'

In the below subcases differnt choices of parameters for creating the result dataframe manually are shown. Note that not all possible combination of paramters are shown here.

### Subcase 1: Static + All Result Values

In [26]:
(s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,properties=None # None => All available properties
                                        ,timestamps=[s3s.GetTimeStamps()[1]] # static
                                        )).head(3)

[2026-11-25 16:22:42,634] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:22:42,688] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Only static timestamp 2026-01-15 00:00:00.000 +01:00 is used


[2026-11-25 16:22:42,694] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:22:42,694] INFO in sir3stoolkit.mantle.dataframes: [results] No properties given → using ALL result properties for ObjectTypes.Pipe.


[2026-11-25 16:22:42,697] INFO in sir3stoolkit.mantle.dataframes: [results] Using 82 result properties.


[2026-11-25 16:22:42,697] INFO in sir3stoolkit.mantle.dataframes: [results] 1 timestamp(s), 572 element(s), 82 properties resolved (0.1 s).


... (8 outputs left out)

[2026-11-25 16:23:34,418] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.8 s).


[2026-11-25 16:23:34,491] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:23:34,499] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (1, 16588) (51.9 s in total)


tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                                A   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                                0.0           

tk                                                                          \
name                                                                         
end_nodes                                                                    
property                            DTTR   DWVERL DWVERLABS IAKTIV IRTRENN   
timestamp                                                                    
2026-01-15 00:00:00.000 +01:00  0.053661  26.1929   2.61929    0.0     0.0   

tk                                                                     \
name                                                                    
end_nodes                                                               
property                             JV                MVEC    PDAMPF   
timestamp                                                               
2026-01-15 00:00:00.000 +01:00  0.08454  27.24994\t27.24994  0.577745   

tk                                        ...  \
name                                      ...   
end_nodes                                 ...   
property                             PHR  ...   
timestamp                                 ...   
2026-01-15 00:00:00.000 +01:00  0.008454  ...   

tk                                                                    4622768628845952753  \
name                                                            Rohr R-H-RATHAUS R-KAS-04   
end_nodes                      ('5761893596493624260', '5007908863305915354', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                      0\t0.05199681           

tk                                                                          \
name                                                                         
end_nodes                                                                    
property                                TVEC       VAV        VI        VK   
timestamp                                                                    
2026-01-15 00:00:00.000 +01:00  60\t59.81641  0.160266  0.160274  0.160259   

tk                                                                   
name                                                                 
end_nodes                                                            
property                       VOLDA  WVL ZAUS ZEIN            ZVEC  
timestamp                                                            
2026-01-15 00:00:00.000 +01:00   0.0  0.0  0.0  0.0  210.14\t210.97  

[1 rows x 16588 columns]

Note that a multi index on the columns is created, holding the most important model_data.
- Level 0: tk (device ID)
- Level 1: name (device name)
- Level 2: end_nodes (tuple of connected node IDs as string)
- Level 3: property (result name)

As can be seen a lot of 99999.0 placeholder values are inserted, because we attempt to access many result values that SIR Calc can calculate but are not requested for this model. Therefore using all available result properties(properties=None), is generally not recommended.

### Subcase 2: All Timestamps + Specific Values

In [27]:
df_pipes_2 = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                                    ,properties=["DTTR", "PHR"] # Specify result properties of interest
                                                    ,timestamps=None # None => All available simulation timestamps
                                                    )

[2026-11-25 16:23:34,554] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:23:34,623] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] No timestamps were given. Checking available simulation timestamps (SIR3S_Model.GetTimeStamps()[0]).


[2026-11-25 16:23:34,623] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] 13 simulation timestamp(s) are available.


[2026-11-25 16:23:34,623] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 13 timestamps: ['2026-01-15 00:00:00.000 +01:00', '2026-01-15 02:00:00.000 +01:00', '2026-01-15 04:00:00.000 +01:00', '2026-01-15 06:00:00.000 +01:00', '2026-01-15 08:00:00.000 +01:00', '2026-01-15 10:00:00.000 +01:00', '2026-01-15 12:00:00.000 +01:00', '2026-01-15 14:00:00.000 +01:00', '2026-01-15 16:00:00.000 +01:00', '2026-01-15 18:00:00.000 +01:00', '2026-01-15 20:00:00.000 +01:00', '2026-01-15 22:00:00.000 +01:00', '2026-01-16 00:00:00.000 +01:00']


[2026-11-25 16:23:34,629] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:23:34,630] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties.


... (8 outputs left out)

[2026-11-25 16:23:36,573] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.


[2026-11-25 16:23:36,573] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:23:36,638] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:23:36,644] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (13, 1144) (2.1 s in total)


In [28]:
df_pipes_2.head(3)

tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.053661           
2026-01-15 02:00:00.000 +01:00                                           0.057855           
2026-01-15 04:00:00.000 +01:00                                           0.052366           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008454   
2026-01-15 02:00:00.000 +01:00  0.007312   
2026-01-15 04:00:00.000 +01:00  0.008862   

tk                                                                    4917696182352390826  \
name                                                                 Rohr R-HS-01 R-HS-00   
end_nodes                      ('5312042414342987450', '4876208276983667420', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054500           
2026-01-15 02:00:00.000 +01:00                                           0.058762           
2026-01-15 04:00:00.000 +01:00                                           0.053184           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008556   
2026-01-15 02:00:00.000 +01:00  0.007411   
2026-01-15 04:00:00.000 +01:00  0.008965   

tk                                                                    4903694188476127486  \
name                                                                 Rohr V-HS-01 V-HS-02   
end_nodes                      ('5141021320509852354', '4691128234547286647', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054099           
2026-01-15 02:00:00.000 +01:00                                           0.058318           
2026-01-15 04:00:00.000 +01:00                                           0.052782           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008323   
2026-01-15 02:00:00.000 +01:00  0.007201   
2026-01-15 04:00:00.000 +01:00  0.008728   

tk                                                                    5471723299187690823  \
name                                                                 Rohr R-HS-02 R-HS-01   
end_nodes                      ('5391191983276320300', '5312042414342987450', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054944           
2026-01-15 02:00:00.000 +01:00                                           0.059231           
2026-01-15 04:00:00.000 +01:00                         

We can access individual values as follows, here for the first supply pipe of the main street, looked up by its ID reference.

In [29]:
pipe_tk = s3s.GetTkFromIDReference("VL-HS-01", s3s.ObjectTypes.Pipe)
df_pipes_2.loc[
    "2026-01-15 00:00:00.000 +01:00",
    (pipe_tk, slice(None), slice(None), "DTTR")
].item()

0.053661178797483444

### Subcase 3: Specific Timestamps + Specific Values + Without Names and End Nodes

We can specify the list of timestamps to include either as a list of str timestamps ...

With include_name_and_end_nodes=False the name and end_nodes column levels are left out - this saves two API calls per element, which matters for large models.

In [30]:
(s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,properties=["DTTR", "PHR"]
                                        ,timestamps=['2026-01-15 00:00:00.000 +01:00', '2026-01-15 06:00:00.000 +01:00', '2026-01-15 22:00:00.000 +01:00'] 
                                        ,include_name_and_end_nodes=False
                                        )).head(3)

[2026-11-25 16:23:36,699] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:23:36,761] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 3 timestamps: ['2026-01-15 00:00:00.000 +01:00', '2026-01-15 06:00:00.000 +01:00', '2026-01-15 22:00:00.000 +01:00']


[2026-11-25 16:23:36,761] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:23:36,761] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties.


[2026-11-25 16:23:36,761] INFO in sir3stoolkit.mantle.dataframes: [results] 3 timestamp(s), 572 element(s), 2 properties resolved (0.1 s).


[2026-11-25 16:23:36,761] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values... (bulk=False)


... (6 outputs left out)

[2026-11-25 16:23:37,224] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.


[2026-11-25 16:23:37,224] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:23:37,224] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (3, 1144) (0.5 s in total)


tk                             5448887806005001177            \
property                                      DTTR       PHR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00            0.053661  0.008454   
2026-01-15 06:00:00.000 +01:00            0.033612  0.020693   
2026-01-15 22:00:00.000 +01:00            0.044065  0.012375   

tk                             4917696182352390826            \
property                                      DTTR       PHR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00            0.054500  0.008556   
2026-01-15 06:00:00.000 +01:00            0.034373  0.020755   
2026-01-15 22:00:00.000 +01:00            0.044750  0.012482   

tk                             4903694188476127486            \
property                                      DTTR       PHR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00            0.054099  0.008323   
2026-01-15 06:00:00.000 +01:00            0.033869  0.020388   
2026-01-15 22:00:00.000 +01:00            0.044440  0.012174   

tk                             5471723299187690823            \
property                                      DTTR       PHR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00            0.054944  0.008424   
2026-01-15 06:00:00.000 +01:00            0.034636  0.020453   
2026-01-15 22:00:00.000 +01:00            0.045130  0.012281   

tk                             5385707832086838233            ...  \
property                                      DTTR       PHR  ...   
timestamp                                                     ...   
2026-01-15 00:00:00.000 +01:00            0.041497  0.018293  ...   
2026-01-15 06:00:00.000 +01:00            0.025622  0.046314  ...   
2026-01-15 22:00:00.000 +01:00            0.034296  0.026499  ...   

tk                             5053463918914261869            \
property                                      DTTR       PHR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00            0.030947  0.009542   
2026-01-15 06:00:00.000 +01:00            0.013407  0.050840   
2026-01-15 22:00:00.000 +01:00            0.031111  0.009442   

tk                             5656894590403112932            \
property                                      DTTR       PHR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00            0.045050  0.008348   
2026-01-15 06:00:00.000 +01:00            0.019328  0.045025   
2026-01-15 22:00:00.000 +01:00            0.045247  0.008275   

tk                             5251036523227278348            \
property                                      DTTR       PHR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00            0.045703  0.008229   
2026-01-15 06:00:00.000 +01:00            0.019750  0.044062   
2026-01-15 22:00:00.000 +01:00            0.045906  0.008156   

tk                             4808966622748700001            \
property                                      DTTR       PHR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00            0.051243  0.003911   
2026-01-15 06:00:00.000 +01:00            0.021840  0.021378   
2026-01-15 22:00:00.000 +01:00            0.051460  0.003878   

tk                             4622768628845952753            
property                                      DTTR       PHR  
timestamp                                                     
2026-01-15 00:00:00.000 +01:00            0.051997  0.003854  
2026-01-15 06:00:00.000 +01:00            0.022320  0.020918  
2026-01-15 22:00:00.000 +01:00            0.052220  0.003821  

[3 rows x 1144 columns]

... or as a list of int indices.

In [31]:
simulation_timestamps=s3s.GetTimeStamps()[0]

In [32]:
for idx, timestamp in enumerate(simulation_timestamps):
    print(idx, timestamp)

0 2026-01-15 00:00:00.000 +01:00
1 2026-01-15 02:00:00.000 +01:00
2 2026-01-15 04:00:00.000 +01:00
3 2026-01-15 06:00:00.000 +01:00
4 2026-01-15 08:00:00.000 +01:00
5 2026-01-15 10:00:00.000 +01:00
6 2026-01-15 12:00:00.000 +01:00
7 2026-01-15 14:00:00.000 +01:00
8 2026-01-15 16:00:00.000 +01:00
9 2026-01-15 18:00:00.000 +01:00
10 2026-01-15 20:00:00.000 +01:00
11 2026-01-15 22:00:00.000 +01:00
12 2026-01-16 00:00:00.000 +01:00


In [33]:
(s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,properties=["DTTR", "PHR"]
                                        ,timestamps=[0, 5, -2] 
                                        )).head(3)

[2026-11-25 16:23:37,335] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:23:37,404] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 3 timestamps: ['2026-01-15 00:00:00.000 +01:00', '2026-01-15 10:00:00.000 +01:00', '2026-01-15 22:00:00.000 +01:00']


[2026-11-25 16:23:37,404] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:23:37,404] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties.


[2026-11-25 16:23:37,404] INFO in sir3stoolkit.mantle.dataframes: [results] 3 timestamp(s), 572 element(s), 2 properties resolved (0.1 s).


[2026-11-25 16:23:37,404] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values... (bulk=False)


... (7 outputs left out)

[2026-11-25 16:23:37,854] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:23:37,921] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:23:37,923] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (3, 1144) (0.6 s in total)


tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.053661           
2026-01-15 10:00:00.000 +01:00                                           0.040688           
2026-01-15 22:00:00.000 +01:00                                           0.044065           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008454   
2026-01-15 10:00:00.000 +01:00  0.014258   
2026-01-15 22:00:00.000 +01:00  0.012375   

tk                                                                    4917696182352390826  \
name                                                                 Rohr R-HS-01 R-HS-00   
end_nodes                      ('5312042414342987450', '4876208276983667420', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054500           
2026-01-15 10:00:00.000 +01:00                                           0.041612           
2026-01-15 22:00:00.000 +01:00                                           0.044750           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008556   
2026-01-15 10:00:00.000 +01:00  0.014355   
2026-01-15 22:00:00.000 +01:00  0.012482   

tk                                                                    4903694188476127486  \
name                                                                 Rohr V-HS-01 V-HS-02   
end_nodes                      ('5141021320509852354', '4691128234547286647', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054099           
2026-01-15 10:00:00.000 +01:00                                           0.040966           
2026-01-15 22:00:00.000 +01:00                                           0.044440           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008323   
2026-01-15 10:00:00.000 +01:00  0.014070   
2026-01-15 22:00:00.000 +01:00  0.012174   

tk                                                                    5471723299187690823  \
name                                                                 Rohr R-HS-02 R-HS-01   
end_nodes                      ('5391191983276320300', '5312042414342987450', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054944           
2026-01-15 10:00:00.000 +01:00                                           0.041896           
2026-01-15 22:00:00.000 +01:00                         

### Subcase 4: Specific Timestamps + Specific Values + Filtering

If we want to filter for more than just the container, we have to filter tks outside the function and then pass them. Works same way as for model_data.

In [34]:
all_tks = s3s.GetTksofElementType(s3s.ObjectTypes.Pipe)

In [35]:
len(all_tks)

572

In [36]:
good_tks = []

In [37]:
for tk in all_tks:
    if s3s.GetValue(tk, "Fkcont")[0] == main_container:
        if s3s.GetValue(tk, "bz.Irtrenn")[0] == "0": # Rohr nicht getrennt
            good_tks.append(tk)

In [38]:
len(good_tks)

572

In [39]:
(s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,tks=good_tks # here we specify that not all available tks should be used
                                        ,properties=["DTTR", "PHR"]
                                        ,timestamps=[0, 5, -2] 
                                        )).head(3)

[2026-11-25 16:23:38,055] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:23:38,123] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 3 timestamps: ['2026-01-15 00:00:00.000 +01:00', '2026-01-15 10:00:00.000 +01:00', '2026-01-15 22:00:00.000 +01:00']


[2026-11-25 16:23:38,123] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:23:38,130] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] 572 tks remain after filtering for given tks.


[2026-11-25 16:23:38,132] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties.


[2026-11-25 16:23:38,132] INFO in sir3stoolkit.mantle.dataframes: [results] 3 timestamp(s), 572 element(s), 2 properties resolved (0.1 s).


... (8 outputs left out)

[2026-11-25 16:23:38,610] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:23:38,673] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:23:38,678] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (3, 1144) (0.6 s in total)


tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.053661           
2026-01-15 10:00:00.000 +01:00                                           0.040688           
2026-01-15 22:00:00.000 +01:00                                           0.044065           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008454   
2026-01-15 10:00:00.000 +01:00  0.014258   
2026-01-15 22:00:00.000 +01:00  0.012375   

tk                                                                    4917696182352390826  \
name                                                                 Rohr R-HS-01 R-HS-00   
end_nodes                      ('5312042414342987450', '4876208276983667420', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054500           
2026-01-15 10:00:00.000 +01:00                                           0.041612           
2026-01-15 22:00:00.000 +01:00                                           0.044750           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008556   
2026-01-15 10:00:00.000 +01:00  0.014355   
2026-01-15 22:00:00.000 +01:00  0.012482   

tk                                                                    4903694188476127486  \
name                                                                 Rohr V-HS-01 V-HS-02   
end_nodes                      ('5141021320509852354', '4691128234547286647', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054099           
2026-01-15 10:00:00.000 +01:00                                           0.040966           
2026-01-15 22:00:00.000 +01:00                                           0.044440           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2026-01-15 00:00:00.000 +01:00  0.008323   
2026-01-15 10:00:00.000 +01:00  0.014070   
2026-01-15 22:00:00.000 +01:00  0.012174   

tk                                                                    5471723299187690823  \
name                                                                 Rohr R-HS-02 R-HS-01   
end_nodes                      ('5391191983276320300', '5312042414342987450', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.054944           
2026-01-15 10:00:00.000 +01:00                                           0.041896           
2026-01-15 22:00:00.000 +01:00                         

### Subcase 5: Specific Vectorized results + Specific Timestamps

For pipes we have interior point result values. These are usually found as vectors with the postfix VEC.

In [40]:
available_result_props=s3s.GetResultProperties_from_elementType(s3s.ObjectTypes.Pipe,False)

In [41]:
available_result_props

['A',
 'ACALC',
 'CPI',
 'CPK',
 'DH',
 'DP',
 'DRAGRED',
 'DRAKONZ',
 'DSI',
 'DSK',
 'DTTR',
 'DWVERL',
 'DWVERLABS',
 'ETAAV',
 'FS',
 'HR',
 'HVEC',
 'IAKTIV',
 'IRTRENN',
 'JV',
 'JV2',
 'LAMBDA',
 'LECKEINAUS',
 'LECKMENGE',
 'LECKORT',
 'LINEPACK',
 'LINEPACKGEOM',
 'LINEPACKRATE',
 'MAINELEMENT',
 'MAV',
 'MI',
 'MK',
 'MKOND',
 'MMAX_INST',
 'MMIN_INST',
 'MVEC',
 'MVECMAX_INST',
 'MVECMIN_INST',
 'PAV',
 'PDAMPF',
 'PHR',
 'PHVEC',
 'PMAX',
 'PMIN',
 'PR',
 'PVEC',
 'PVECMAX_INST',
 'PVECMIN_INST',
 'QI2',
 'QK2',
 'QMAV',
 'QMI',
 'QMK',
 'QMMAX_INST',
 'QMMIN_INST',
 'QMVEC',
 'QSVB',
 'RHOAV',
 'RHOI',
 'RHOK',
 'RHOVEC',
 'SVEC',
 'TAV',
 'TI',
 'TK',
 'TTRVEC',
 'TVEC',
 'TVECMAX_INST',
 'TVECMIN_INST',
 'VAV',
 'VI',
 'VK',
 'VMAX_INST',
 'VMIN_INST',
 'VOLDA',
 'WALTERI',
 'WALTERK',
 'WVL',
 'ZAUS',
 'ZEIN',
 'ZHKNR',
 'ZVEC']

In [42]:
available_result_vector_props=[available_result_props for available_result_props in available_result_props if "VEC" in available_result_props]

In [43]:
available_result_vector_props

['HVEC',
 'MVEC',
 'MVECMAX_INST',
 'MVECMIN_INST',
 'PHVEC',
 'PVEC',
 'PVECMAX_INST',
 'PVECMIN_INST',
 'QMVEC',
 'RHOVEC',
 'SVEC',
 'TTRVEC',
 'TVEC',
 'TVECMAX_INST',
 'TVECMIN_INST',
 'ZVEC']

In [44]:
df_pipes = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,properties=["TTRVEC", "MVEC", "DTTR"]
                                        ,timestamps=[0, 5, -2] 
                                        )

[2026-11-25 16:23:38,748] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:23:38,814] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 3 timestamps: ['2026-01-15 00:00:00.000 +01:00', '2026-01-15 10:00:00.000 +01:00', '2026-01-15 22:00:00.000 +01:00']


[2026-11-25 16:23:38,814] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:23:38,814] INFO in sir3stoolkit.mantle.dataframes: [results] Using 3 result properties.


[2026-11-25 16:23:38,814] INFO in sir3stoolkit.mantle.dataframes: [results] 3 timestamp(s), 572 element(s), 3 properties resolved (0.1 s).


[2026-11-25 16:23:38,814] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values... (bulk=False)


... (6 outputs left out)

[2026-11-25 16:23:39,642] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.


[2026-11-25 16:23:39,642] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:23:39,721] INFO in sir3stoolkit.mantle.dataframes: [results] Names/end nodes of 572 element(s) looked up (0.1 s).


[2026-11-25 16:23:39,724] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (3, 1716) (1.0 s in total)


In [45]:
df_pipes.head(3)

tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                      0\t0.05366118           
2026-01-15 10:00:00.000 +01:00                                      0\t0.04068783           
2026-01-15 22:00:00.000 +01:00                                      0\t0.04406466           

tk                                                            \
name                                                           
end_nodes                                                      
property                                      MVEC      DTTR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00  27.24994\t27.24994  0.053661   
2026-01-15 10:00:00.000 +01:00  35.68641\t35.68641  0.040688   
2026-01-15 22:00:00.000 +01:00  33.18446\t33.18446  0.044065   

tk                                                                    4917696182352390826  \
name                                                                 Rohr R-HS-01 R-HS-00   
end_nodes                      ('5312042414342987450', '4876208276983667420', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                               0.5864736\t0.6409737           
2026-01-15 10:00:00.000 +01:00                               0.4477745\t0.4893861           
2026-01-15 22:00:00.000 +01:00                               0.4815553\t0.5263053           

tk                                                            \
name                                                           
end_nodes                                                      
property                                      MVEC      DTTR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00  27.24994\t27.24994  0.054500   
2026-01-15 10:00:00.000 +01:00  35.68641\t35.68641  0.041612   
2026-01-15 22:00:00.000 +01:00  33.18446\t33.18446  0.044750   

tk                                                                    4903694188476127486  \
name                                                                 Rohr V-HS-01 V-HS-02   
end_nodes                      ('5141021320509852354', '4691128234547286647', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                              0.05366118\t0.1077601           
2026-01-15 10:00:00.000 +01:00                             0.04068783\t0.08165419           
2026-01-15 22:00:00.000 +01:00                             0.04406466\t0.08850425           

tk                                                            \
name                                                           
end_nodes                                                      
property                                      MVEC      DTTR   
timestamp                                                      
2026-01-15 00:00:00.000 +01:00  27.02984\t27.02984  0.054099   
2026-01-15 10:00:00.000 +01:00  35.44429\t35.44429  0.040966   
2026-01-15 22:00:00.000 +01:00  32.90487\t32.90487  0.044440   

tk                                                                    5471723299187690823  \
name                                                                 Rohr R-HS-02 R-HS-01   
end_nodes                      ('5391191

As can be seen, the vectorized result values (one value per interior point) are written as tab-separated strings (vec_format="str", the default). For better access we can split them into the columns <property>_start, <property>_end and <property>_sequence (a tuple of all interior point values) using [add_interior_points_to_start_end_sequence()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.add_interior_points_to_start_end_sequence). Alternatively, generate_element_results_dataframe(vec_format="multiindex") returns one float column per interior point directly, see [Tutorial 57](https://3sconsult.github.io/sir3stoolkit/examples.html#ttu057).

In [46]:
df_pipes_interior_points = s3s.add_interior_points_to_start_end_sequence(df_pipes)

In [47]:
df_pipes_interior_points.head(3)

tk                                                                    5448887806005001177  \
name                                                                 Rohr V-HS-00 V-HS-01   
end_nodes                      ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                                     TTRVEC_start   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                                0.0           
2026-01-15 10:00:00.000 +01:00                                                0.0           
2026-01-15 22:00:00.000 +01:00                                                0.0           

tk                                                                       \
name                                                                      
end_nodes                                                                 
property                       TTRVEC_end    TTRVEC_sequence MVEC_start   
timestamp                                                                 
2026-01-15 00:00:00.000 +01:00   0.053661  (0.0, 0.05366118)   27.24994   
2026-01-15 10:00:00.000 +01:00   0.040688  (0.0, 0.04068783)   35.68641   
2026-01-15 22:00:00.000 +01:00   0.044065  (0.0, 0.04406466)   33.18446   

tk                                                                        \
name                                                                       
end_nodes                                                                  
property                        MVEC_end         MVEC_sequence      DTTR   
timestamp                                                                  
2026-01-15 00:00:00.000 +01:00  27.24994  (27.24994, 27.24994)  0.053661   
2026-01-15 10:00:00.000 +01:00  35.68641  (35.68641, 35.68641)  0.040688   
2026-01-15 22:00:00.000 +01:00  33.18446  (33.18446, 33.18446)  0.044065   

tk                                                                    4917696182352390826  \
name                                                                 Rohr R-HS-01 R-HS-00   
end_nodes                      ('5312042414342987450', '4876208276983667420', '-1', '-1')   
property                                                                     TTRVEC_start   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.586474           
2026-01-15 10:00:00.000 +01:00                                           0.447775           
2026-01-15 22:00:00.000 +01:00                                           0.481555           

tk                                                                 ...  \
name                                                               ...   
end_nodes                                                          ...   
property                       TTRVEC_end         TTRVEC_sequence  ...   
timestamp                                                          ...   
2026-01-15 00:00:00.000 +01:00   0.640974  (0.5864736, 0.6409737)  ...   
2026-01-15 10:00:00.000 +01:00   0.489386  (0.4477745, 0.4893861)  ...   
2026-01-15 22:00:00.000 +01:00   0.526305  (0.4815553, 0.5263053)  ...   

tk                                                                    4808966622748700001  \
name                                                            Rohr V-KAS-04 V-H-RATHAUS   
end_nodes                      ('5621769139637743034', '5585320695144407718', '-1', '-1')   
property                                                                         MVEC_end   
timestamp                                                                                   
2026-01-15 00:00:00.000 +01:00                                           0.446371           
2026-01-15 10:00:00.000 +01:00                                           1.146122           
2026-01-15 22:00:00.000 +01:00                          

We can access individual values as follows.

In [48]:
df_pipes_interior_points.loc[
    "2026-01-15 00:00:00.000 +01:00",
    (pipe_tk, slice(None), slice(None), "MVEC_sequence")
].item()


(27.24994, 27.24994)

We can also collapse the three time rows into one tuple.

In [49]:
df_pipes_simplified, timestamp_to_tuple_index = s3s.convert_rows_to_single_tuple_row(df_pipes_interior_points)

[2026-11-25 16:23:40,439] INFO in sir3stoolkit.mantle.dataframes: [rows_to_tuple_row] Collapsing dataframe rows into tuple row...


[2026-11-25 16:23:40,556] INFO in sir3stoolkit.mantle.dataframes: [rows_to_tuple_row] Done. Shape: (1, 4004)


In [50]:
df_pipes_simplified

tk                                               5448887806005001177  \
name                                            Rohr V-HS-00 V-HS-01   
end_nodes ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                TTRVEC_start   
0                                            (0.0, 0.0, 0.0)           

tk                                               \
name                                              
end_nodes                                         
property                             TTRVEC_end   
0          (0.05366118, 0.04068783, 0.04406466)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                     TTRVEC_sequence   
0          ((0.0, 0.05366118), (0.0, 0.04068783), (0.0, 0...   

tk                                                                         \
name                                                                        
end_nodes                                                                   
property                       MVEC_start                        MVEC_end   
0          (27.24994, 35.68641, 33.18446)  (27.24994, 35.68641, 33.18446)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                       MVEC_sequence   
0          ((27.24994, 27.24994), (35.68641, 35.68641), (...   

tk                                               \
name                                              
end_nodes                                         
property                                   DTTR   
0          (0.05366118, 0.04068783, 0.04406466)   

tk                                               4917696182352390826  \
name                                            Rohr R-HS-01 R-HS-00   
end_nodes ('5312042414342987450', '4876208276983667420', '-1', '-1')   
property                                                TTRVEC_start   
0                          (0.5864736, 0.4477745, 0.4815553)           

tk                                            \
name                                           
end_nodes                                      
property                          TTRVEC_end   
0          (0.6409737, 0.4893861, 0.5263053)   

tk                                                            ...  \
name                                                          ...   
end_nodes                                                     ...   
property                                     TTRVEC_sequence  ...   
0          ((0.5864736, 0.6409737), (0.4477745, 0.4893861...  ...   

tk                                               4808966622748700001  \
name                                       Rohr V-KAS-04 V-H-RATHAUS   
end_nodes ('5621769139637743034', '5585320695144407718', '-1', '-1')   
property                                                    MVEC_end   
0                           (0.4463711, 1.146122, 0.4444616)           

tk                                                            \
name                                                           
end_nodes                                                      
property                                       MVEC_sequence   
0          ((0.4463711, 0.4463711), (1.146122, 1.146122),...   

tk                                               \
name                                              
end_nodes                                         
property                                   DTTR   
0          (0.05124274, 0.01981596, 0.05145952)   

tk                                               4622768628845952753  \
name                                       Rohr R-H-RATHAUS R-KAS-04   
end_nodes ('5761893596493624260', '5007908863305915354', '-1', '-1') 

For every result value we now have a tuple with each entry corresponding to one simulation timestamp.

In [51]:
df_pipes_simplified.loc[
    0,
    (pipe_tk, slice(None), slice(None), "DTTR")
].item()

(0.05366118, 0.04068783, 0.04406466)

We can access individual values via the dict, that was returned.

In [52]:
df_pipes_simplified.loc[
    0,
    (pipe_tk, slice(None), slice(None), "DTTR")
].item()[timestamp_to_tuple_index['2026-01-15 00:00:00.000 +01:00']]

0.05366118

For vectorized result properties (pipe interior points) we will receive a tuple of tuples. With the time tuple as the higher level one.

In [53]:
df_pipes_simplified.loc[
    0,
    (pipe_tk, slice(None), slice(None), "MVEC_sequence")
].item()

((27.24994, 27.24994), (35.68641, 35.68641), (33.18446, 33.18446))

In [54]:
df_pipes_simplified.loc[
    0,
    (pipe_tk, slice(None), slice(None), "MVEC_sequence")
].item()[timestamp_to_tuple_index['2026-01-15 00:00:00.000 +01:00']]

(27.24994, 27.24994)

## Merge model_data and Result Dataframes (for one timestamp)

### Create Dataframes - For pipes

#### model_data

In [55]:
df_pipes_model_data = s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Pipe
                                                            ,properties=["L", "DN"]
                                                            ,geometry=True
                                                            )

[2026-11-25 16:23:40,641] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:23:40,641] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:23:40,646] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 2 model_data properties.


[2026-11-25 16:23:40,646] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['L', 'DN'], geometry...


[2026-11-25 16:23:40,758] WARNING in sir3stoolkit.mantle.dataframes: [model_data] Spatial Reference Identifier (SRID) not defined in model. DataFrame cannot be transformed to GeoDataFrame but geometry column can be created independently of SRID. Returning regular DataFrame with a geometry column.


[2026-11-25 16:23:40,758] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (572, 4)


In [56]:
df_pipes_model_data.head(3)

,tk,L,DN,geometry
0,5448887806005001177,100,250,"LINESTRING Z (100 1000 198.08999633789062, 200..."
1,4917696182352390826,100,250,"LINESTRING Z (204 996 197.9499969482422, 104 9..."
2,4903694188476127486,100,250,"LINESTRING Z (200 1000 197.9499969482422, 300 ..."


#### Results

In [57]:
df_pipes_simplified.head(3)

tk                                               5448887806005001177  \
name                                            Rohr V-HS-00 V-HS-01   
end_nodes ('5431799190353317880', '5141021320509852354', '-1', '-1')   
property                                                TTRVEC_start   
0                                            (0.0, 0.0, 0.0)           

tk                                               \
name                                              
end_nodes                                         
property                             TTRVEC_end   
0          (0.05366118, 0.04068783, 0.04406466)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                     TTRVEC_sequence   
0          ((0.0, 0.05366118), (0.0, 0.04068783), (0.0, 0...   

tk                                                                         \
name                                                                        
end_nodes                                                                   
property                       MVEC_start                        MVEC_end   
0          (27.24994, 35.68641, 33.18446)  (27.24994, 35.68641, 33.18446)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                       MVEC_sequence   
0          ((27.24994, 27.24994), (35.68641, 35.68641), (...   

tk                                               \
name                                              
end_nodes                                         
property                                   DTTR   
0          (0.05366118, 0.04068783, 0.04406466)   

tk                                               4917696182352390826  \
name                                            Rohr R-HS-01 R-HS-00   
end_nodes ('5312042414342987450', '4876208276983667420', '-1', '-1')   
property                                                TTRVEC_start   
0                          (0.5864736, 0.4477745, 0.4815553)           

tk                                            \
name                                           
end_nodes                                      
property                          TTRVEC_end   
0          (0.6409737, 0.4893861, 0.5263053)   

tk                                                            ...  \
name                                                          ...   
end_nodes                                                     ...   
property                                     TTRVEC_sequence  ...   
0          ((0.5864736, 0.6409737), (0.4477745, 0.4893861...  ...   

tk                                               4808966622748700001  \
name                                       Rohr V-KAS-04 V-H-RATHAUS   
end_nodes ('5621769139637743034', '5585320695144407718', '-1', '-1')   
property                                                    MVEC_end   
0                           (0.4463711, 1.146122, 0.4444616)           

tk                                                            \
name                                                           
end_nodes                                                      
property                                       MVEC_sequence   
0          ((0.4463711, 0.4463711), (1.146122, 1.146122),...   

tk                                               \
name                                              
end_nodes                                         
property                                   DTTR   
0          (0.05124274, 0.01981596, 0.05145952)   

tk                                               4622768628845952753  \
name                                       Rohr R-H-RATHAUS R-KAS-04   
end_nodes ('5761893596493624260', '5007908863305915354', '-1', '-1') 

### Merge

In [58]:
df_pipes_simplified.columns = df_pipes_simplified.columns.droplevel([1, 2])
df_pipes_simplified = df_pipes_simplified.T.unstack(level=0).T
df_pipes_simplified = df_pipes_simplified.droplevel(0, axis=0)
df_pipes = df_pipes_model_data.merge(on="tk",
                    how="outer",
                    right=df_pipes_simplified)

In [59]:
df_pipes.head(3)

,tk,L,DN,geometry,DTTR,MVEC_end,MVEC_sequence,MVEC_start,TTRVEC_end,TTRVEC_sequence,TTRVEC_start
0,4612274316321154308,15,25,"LINESTRING Z (900 1270 208.30999755859375, 915...","(0.01794169, 0.01638285, 0.01422273)","(0.163439, 0.177749, 0.2061498)","((0.163439, 0.163439), (0.177749, 0.177749), (...","(0.163439, 0.177749, 0.2061498)","(0.5374579, 0.4170788, 0.440981)","((0.5195163, 0.5374579), (0.4006959, 0.4170788...","(0.5195163, 0.4006959, 0.4267583)"
1,4614940901611765838,15,25,"LINESTRING Z (300 1220 198.0399932861328, 315 ...","(0.01731308, 0.01572273, 0.01368067)","(0.1693296, 0.1851594, 0.2142741)","((0.1693296, 0.1693296), (0.1851594, 0.1851594...","(0.1693296, 0.1851594, 0.2142741)","(0.2058259, 0.1663528, 0.1671905)","((0.1885128, 0.2058259), (0.1506301, 0.1663528...","(0.1885128, 0.1506301, 0.1535098)"
2,4618102496224966047,15,25,"LINESTRING Z (300 830 200.32000732421875, 315 ...","(0.03966657, 0.03610635, 0.03142544)","(0.07391112, 0.08063656, 0.09328599)","((0.07391112, 0.07391112), (0.08063656, 0.0806...","(0.07391112, 0.08063656, 0.09328599)","(0.2201724, 0.1840432, 0.1776151)","((0.1805058, 0.2201724), (0.1479369, 0.1840432...","(0.1805058, 0.1479369, 0.1461897)"


To get this dataframe without having to perform all intermediate steps, use [generate_element_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_element_dataframe), described in Tutorial 52.